# ECI data pipeline

Single-notebook, refreshable data prep for the ECI Bayesian model. Run all
cells top-to-bottom to:

1. **Fetch** Epoch AI's bulk benchmark ZIP (CC-BY) into a dated snapshot folder.
2. **Inventory** what's inside and diff against the previous snapshot.
3. **Load + normalize** scores into a long-format table.
4. **Canonicalize** benchmark and model names.
5. **Dedup** exact duplicates (and resolve disagreements by `max`).
6. **Attach metadata** (category, lower_bound) from a human-maintained file.
7. **Filter** (optional curated exclusions; off by default).
8. **Humans** — apply the canonicalization map to `data/curated/human_baselines.csv`.
9. **Assemble** the final `benchmarks_merged.csv` and a `pipeline_report.md` provenance summary.

Outputs land in `data/pipeline/output/`. When happy, manually copy:

```
cp output/benchmarks_merged.csv ../processed/benchmarks_merged.csv
cp output/human_baselines.csv   ../curated/human_baselines.csv
```

then re-run `python run.py`.


## 00 — Setup

Imports, paths, parameters.

In [1]:
import json
import hashlib
import shutil
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Notebook is expected to run from data/pipeline/
PIPELINE_DIR = Path.cwd()
if PIPELINE_DIR.name != "pipeline":
    print(f"WARN: cwd is {PIPELINE_DIR}, expected to end in /pipeline. Paths may be wrong.")

PROJECT_ROOT    = PIPELINE_DIR.parents[1]
CANONICAL_DIR   = PIPELINE_DIR / "canonical"
SNAPSHOTS_DIR   = PIPELINE_DIR / "snapshots"
INTERMEDIATE_DIR = PIPELINE_DIR / "intermediate"
OUTPUT_DIR      = PIPELINE_DIR / "output"
CURATED_DIR     = PROJECT_ROOT / "data" / "curated"
PROCESSED_DIR   = PROJECT_ROOT / "data" / "processed"

for p in [SNAPSHOTS_DIR, INTERMEDIATE_DIR, OUTPUT_DIR, CANONICAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# --- Parameters ---
EPOCH_ZIP_URL = "https://epoch.ai/data/benchmark_data.zip"

# Off by default: matches the canonical broad-index config from CLAUDE.md.
APPLY_CURATED_EXCLUSIONS = True

# Fallback policy for the (rare) case of disagreeing duplicates on (model, benchmark).
# Exact-row duplicates are always dropped regardless.
DEDUP_POLICY = "max"

# Match data.py's open-interval clip used for the Beta likelihood.
ECI_EPS = 1e-3

print(f"PIPELINE_DIR = {PIPELINE_DIR}")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


PIPELINE_DIR = ~/Desktop/ECI_Bayesian/data/pipeline
PROJECT_ROOT = ~/Desktop/ECI_Bayesian


## 01 — Fetch

Download Epoch's bulk ZIP into `snapshots/YYYY-MM-DD/`. Idempotent within a
day (won't re-download if today's snapshot exists). Writes `provenance.json`
with the URL, sha256, fetch time, license, and attribution.

The three Epoch-missing sources (PRBench Finance, OSUniverse, Lech Mazur)
are not auto-fetched in v1 — drop them into the snapshot folder manually if
you want them included.


In [2]:
today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
snapshot_dir = SNAPSHOTS_DIR / today
epoch_dir    = snapshot_dir / "epoch"
zip_path     = snapshot_dir / "epoch_benchmark_data.zip"
prov_path    = snapshot_dir / "provenance.json"

if snapshot_dir.exists() and prov_path.exists() and any(epoch_dir.rglob("*.csv")):
    print(f"Snapshot already exists at {snapshot_dir} — re-using.")
else:
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {EPOCH_ZIP_URL}")
    urllib.request.urlretrieve(EPOCH_ZIP_URL, zip_path)

    sha = hashlib.sha256(zip_path.read_bytes()).hexdigest()
    size = zip_path.stat().st_size
    print(f"  sha256: {sha}")
    print(f"  size:   {size:,} bytes")

    if epoch_dir.exists():
        shutil.rmtree(epoch_dir)
    epoch_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(epoch_dir)
    n_files = sum(1 for _ in epoch_dir.rglob("*"))
    print(f"  unzipped {n_files} files to {epoch_dir}")

    prov = {
        "url":         EPOCH_ZIP_URL,
        "fetched_at":  datetime.now(timezone.utc).isoformat(),
        "sha256":      sha,
        "byte_size":   size,
        "license":     "CC-BY",
        "attribution": "Epoch AI",
    }
    prov_path.write_text(json.dumps(prov, indent=2))
    print(f"  wrote {prov_path.name}")

LATEST_SNAPSHOT  = snapshot_dir
LATEST_EPOCH_DIR = epoch_dir

print("\nOptional supplementary sources — drop into the snapshot folder manually if available:")
print(f"  {snapshot_dir / 'prbench_finance.csv'}   (Scale SEAL — HTML scrape)")
print(f"  {snapshot_dir / 'osuniverse.csv'}         (agentsea/osuniverse — GitHub)")
print(f"  {snapshot_dir / 'lech_mazur_writing.csv'} (lechmazur/writing — GitHub)")


Snapshot already exists at ~/Desktop/ECI_Bayesian/data/pipeline/snapshots/2026-06-19 — re-using.

Optional supplementary sources — drop into the snapshot folder manually if available:
  ~/Desktop/ECI_Bayesian/data/pipeline/snapshots/2026-06-19/prbench_finance.csv   (Scale SEAL — HTML scrape)
  ~/Desktop/ECI_Bayesian/data/pipeline/snapshots/2026-06-19/osuniverse.csv         (agentsea/osuniverse — GitHub)
  ~/Desktop/ECI_Bayesian/data/pipeline/snapshots/2026-06-19/lech_mazur_writing.csv (lechmazur/writing — GitHub)


## 02 — Inventory

List the CSVs in the latest snapshot with their shape and column names so
you can verify what Epoch shipped this week, and adjust the column-alias
map in section 03 if Epoch renamed something.

Also diffs against the previous snapshot folder if one exists.


In [3]:
print(f"Inventorying {LATEST_EPOCH_DIR}\n")

csv_files = sorted(LATEST_EPOCH_DIR.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s):\n")

for f in csv_files:
    try:
        df = pd.read_csv(f, low_memory=False, nrows=5000)
        print(f"{f.relative_to(LATEST_EPOCH_DIR)}: {len(df):,}+ rows × {len(df.columns)} cols")
        print(f"  cols: {list(df.columns)}")
    except Exception as e:
        print(f"{f.name}: read failed ({e})")
    print()

# Diff vs previous snapshot
prev_snapshots = sorted(
    [p for p in SNAPSHOTS_DIR.iterdir() if p.is_dir() and p != LATEST_SNAPSHOT]
)
if prev_snapshots:
    prev = prev_snapshots[-1]
    print(f"Diff vs previous snapshot ({prev.name}):")
    prev_files = {f.name for f in (prev / "epoch").rglob("*.csv")}
    curr_files = {f.name for f in LATEST_EPOCH_DIR.rglob("*.csv")}
    new  = curr_files - prev_files
    gone = prev_files - curr_files
    if new:  print(f"  new files: {sorted(new)}")
    if gone: print(f"  removed:   {sorted(gone)}")
    if not new and not gone:
        print("  (no file-level changes)")
else:
    print("(no previous snapshot to diff against)")


Inventorying ~/Desktop/ECI_Bayesian/data/pipeline/snapshots/2026-06-19/epoch

Found 54 CSV file(s):

additional_eci_data/eci_benchmark_difficulties_and_slopes.csv: 42+ rows × 5 cols
  cols: ['benchmark_name', 'is_anchor', 'benchmark_release_date', 'edi', 'estimated_slope_scaled']

additional_eci_data/eci_scaling.csv: 1+ rows × 6 cols
  cols: ['a', 'b', 'scaling_anchor1', 'scaling_anchor1_eci', 'scaling_anchor2', 'scaling_anchor2_eci']

adversarial_nli_external.csv: 19+ rows × 16 cols
  cols: ['Model version', 'Score', 'Release date', 'Organization', 'Country', 'Training compute (FLOP)', 'Training compute notes', 'Name', 'R1', 'R2', 'R3', 'Shots', 'Source', 'Source link', 'Notes', 'id']

aider_polyglot_external.csv: 77+ rows × 16 cols
  cols: ['Model version', 'Percent correct', 'Release date', 'Organization', 'Country', 'Training compute (FLOP)', 'Training compute notes', 'Percent using correct edit format', 'Edit format', 'Cost', 'Source', 'Source link', 'Notes', 'Time per case (secon

## 03 — Load + normalize

Epoch ships one CSV per benchmark with the benchmark identity in the
filename and the score column varying per file (`EM`, `Accuracy`,
`mean_score`, `Mean score`, `Score (AVG@5)`, `Pass@1 score`, etc.). The
loader uses an explicit per-file spec map (`FILE_SPEC`):

```
filename → (canonical_benchmark_name, score_column, divisor)
```

`divisor` rescales the column onto `[0, 1]`. Most Epoch columns are
already on that scale even when named like a percentage (`% Score`,
`Overall pass (%)`) — `divisor=1.0` for those. A handful are genuine
percentages or out-of-10 (`Percent correct`, `Global average`, `Mean
score`) and get a real divisor.

Files not in `FILE_SPEC` are listed in `FILE_SKIP` with a reason, or
flagged as unknown. Edit the spec when Epoch adds a new file.


In [4]:
# Per-file spec — adjust when Epoch publishes new files.
# Format: filename: (benchmark_name, score_column, divisor)
FILE_SPEC = {
    # --- External (model-developer reported) ---
    "adversarial_nli_external.csv":    ("Adversarial NLI",        "Score",                  1.0),
    "aider_polyglot_external.csv":     ("Aider Polyglot",         "Percent correct",        100.0),
    "apex_agents_external.csv":        ("APEX Agents",            "Pass@1 score",           1.0),
    "arc_agi_external.csv":            ("ARC-AGI",                "Score",                  1.0),
    "arc_agi_2_external.csv":          ("ARC-AGI-2",              "Score",                  1.0),
    "arc_ai2_external.csv":            ("ARC (AI2)",              "Challenge score",        1.0),
    "balrog_external.csv":             ("BALROG",                 "Average progress",       1.0),
    "bbh_external.csv":                ("BIG-Bench Hard (BBH)",   "Average",                1.0),
    "bool_q_external.csv":             ("BoolQ",                  "Score",                  1.0),
    "cad_eval_external.csv":           ("CAD-Eval",               "Overall pass (%)",       1.0),
    "common_sense_qa_2_external.csv":  ("CSQA2",                  "Score",                  1.0),
    "cybench_external.csv":            ("Cybench",                "Unguided % Solved",      1.0),
    "deepresearchbench_external.csv":  ("DeepResearchBench",      "Average score",          1.0),
    "fictionlivebench_external.csv":   ("Fiction.LiveBench",      "120k token score",       1.0),
    "gsm8k_external.csv":              ("GSM8K",                  "EM",                     1.0),
    "gso_external.csv":                ("GSO-Bench",              "Score OPT@1",            1.0),
    "hella_swag_external.csv":         ("HellaSwag",              "Overall accuracy",       1.0),
    "hle_external.csv":                ("Humanity\'s Last Exam", "Accuracy",               1.0),
    "lambada_external.csv":            ("LAMBADA",                "Score",                  1.0),
    "lech_mazur_writing_external.csv": ("Lech Mazur Writing",     "Mean score",             10.0),
    "live_bench_external.csv":         ("LiveBench",              "Global average",         100.0),
    "metr_time_horizons_external.csv": ("METR Time Horizons",     "average_score",          1.0),
    "mmlu_external.csv":               ("MMLU",                   "EM",                     1.0),
    "open_book_qa_external.csv":       ("OpenBookQA",             "Accuracy",               1.0),
    "os_world_external.csv":           ("OS World (Screenshot)",  "Score",                  100.0),
    "piqa_external.csv":               ("PIQA",                   "Score",                  1.0),
    "posttrainbench_external.csv":     ("PostTrainBench",         "Average (%)",            1.0),
    "science_qa_external.csv":         ("ScienceQA",              "Score",                  1.0),
    "simplebench_external.csv":        ("SimpleBench",            "Score (AVG@5)",          1.0),
    "superglue_external.csv":          ("SuperGLUE",              "Score",                  1.0),
    "terminalbench_external.csv":      ("TerminalBench",          "Accuracy mean",          1.0),
    "the_agent_company_external.csv":  ("The Agent Company",      "% Score",                1.0),
    "trivia_qa_external.csv":          ("TriviaQA",               "EM",                     1.0),
    "video_mme_external.csv":          ("Video-MME",              "Overall (no subtitles)", 1.0),
    "vpct_external.csv":               ("VPCT",                   "Correct",                1.0),
    "weirdml_external.csv":            ("WeirdML",                "Accuracy",               1.0),
    "wino_grande_external.csv":        ("WinoGrande",             "Accuracy",               1.0),

    # --- Internal Epoch evals (use mean_score; stderr available alongside) ---
    "chess_puzzles.csv":               ("Chess Puzzles",          "mean_score",             1.0),
    "frontiermath.csv":                ("FrontierMath",           "mean_score",             1.0),
    "frontiermath_tier_4.csv":         ("FrontierMath Tier 4",    "mean_score",             1.0),
    "gpqa_diamond.csv":                ("GPQA Diamond",           "mean_score",             1.0),
    "math_level_5.csv":                ("MATH Level 5",           "mean_score",             1.0),
    "otis_mock_aime_2024_2025.csv":    ("OTIS Mock AIME 2024-2025","mean_score",            1.0),
    "simpleqa_verified.csv":           ("SimpleQA Verified",      "mean_score",             1.0),
    "swe_bench_verified.csv":          ("SWE-Bench Verified",     "mean_score",             1.0),
}

# Files explicitly excluded with reason. Anything not in FILE_SPEC and not
# in FILE_SKIP gets flagged as "unrecognized".
FILE_SKIP = {
    "epoch_capabilities_index.csv":   "Epoch\'s own ECI output (downstream of this pipeline)",
    "frontierswe_external.csv":       "ranks, not a [0,1] score",
    "webdev_arena_external.csv":      "Elo-style Arena Score, not [0,1]",
    "geobench_external.csv":          "multi-axis composite, scale ambiguous (TODO if useful)",
    "gbaeval_external.csv":           "overall score scale unclear (TODO)",
    "gdpval_external.csv":            "win rate vs other models, not absolute capability",
    "gdp_pdf_external.csv":           "single-model evaluation, scale ambiguous (TODO)",
    "README.md":                      "not a data file",
}

# Standard meta-column rename map (Epoch is consistent across files).
META_RENAME = {
    "Model version":      "model_version",
    "Release date":       "release_date",
    "Organization":       "organization",
    "Source":             "source",
    "Source link":        "source_link",
    "Notes":              "notes",
    "id":                 "epoch_id",
}

frames = []
unrecognized = []

for f in sorted(LATEST_EPOCH_DIR.rglob("*.csv")):
    name = f.name
    if name in FILE_SKIP:
        print(f"  - SKIP {name}: {FILE_SKIP[name]}")
        continue
    if name not in FILE_SPEC:
        unrecognized.append(name)
        continue

    bench_name, score_col, divisor = FILE_SPEC[name]
    try:
        df = pd.read_csv(f, low_memory=False)
    except Exception as e:
        print(f"  ! {name}: read failed: {e}")
        continue

    if score_col not in df.columns:
        print(f"  ! {name}: expected score column {score_col!r} not found; has {list(df.columns)}")
        continue
    if "Model version" not in df.columns:
        print(f"  ! {name}: no 'Model version' column; has {list(df.columns)}")
        continue

    out = pd.DataFrame({
        "model_version": df["Model version"],
        "score":         pd.to_numeric(df[score_col], errors="coerce") / divisor,
        "benchmark":     bench_name,
    })
    for upstream, canonical in META_RENAME.items():
        if upstream in df.columns and canonical not in {"model_version"}:
            out[canonical] = df[upstream]

    out["_source_file"] = name
    out["_source_row"]  = df.index
    out["_score_col"]   = score_col

    # Drop NaN-score and NaN-model rows
    before = len(out)
    out = out.dropna(subset=["model_version", "score"])
    if before != len(out):
        print(f"  + {name}: {len(out):,} rows ({before - len(out)} dropped for NaN)")
    else:
        print(f"  + {name}: {len(out):,} rows")
    frames.append(out)

if unrecognized:
    print(f"\nUnrecognized files (not in FILE_SPEC or FILE_SKIP) — add them or skip them:")
    for n in unrecognized:
        print(f"  ? {n}")

if not frames:
    raise RuntimeError("No frames parsed — check FILE_SPEC.")

scores_long = pd.concat(frames, ignore_index=True, sort=False)
print(f"\nConcatenated: {len(scores_long):,} rows across {scores_long['benchmark'].nunique()} benchmarks")

# Range assertion
out_of_range = scores_long[(scores_long["score"] < 0) | (scores_long["score"] > 1)]
if len(out_of_range):
    print(f"\nFAIL: {len(out_of_range)} scores outside [0, 1]. Sample:")
    print(out_of_range[["model_version", "benchmark", "score", "_source_file"]].head(20).to_string(index=False))
    raise ValueError(
        "Some scores fall outside [0, 1]. Check the divisor in FILE_SPEC for the offending benchmark."
    )

print(f"\nAll {len(scores_long):,} scores in [0, 1].")

out_path = INTERMEDIATE_DIR / "02_scores_long.csv"
scores_long.to_csv(out_path, index=False)
print(f"Wrote {out_path}")


  + adversarial_nli_external.csv: 15 rows (4 dropped for NaN)
  + aider_polyglot_external.csv: 72 rows (5 dropped for NaN)
  + apex_agents_external.csv: 46 rows
  + arc_agi_2_external.csv: 142 rows
  + arc_agi_external.csv: 142 rows (2 dropped for NaN)
  + arc_ai2_external.csv: 134 rows (11 dropped for NaN)
  + balrog_external.csv: 36 rows
  + bbh_external.csv: 88 rows (4 dropped for NaN)
  + bool_q_external.csv: 136 rows (70 dropped for NaN)
  + cad_eval_external.csv: 15 rows
  + chess_puzzles.csv: 47 rows
  + common_sense_qa_2_external.csv: 6 rows (4 dropped for NaN)
  + cybench_external.csv: 22 rows
  + deepresearchbench_external.csv: 22 rows (1 dropped for NaN)
  - SKIP epoch_capabilities_index.csv: Epoch's own ECI output (downstream of this pipeline)
  + fictionlivebench_external.csv: 42 rows (20 dropped for NaN)
  + frontiermath.csv: 101 rows
  + frontiermath_tier_4.csv: 72 rows
  - SKIP frontierswe_external.csv: ranks, not a [0,1] score
  - SKIP gbaeval_external.csv: overall sco

## 03b — RAND biology & chemistry

Epoch's ZIP doesn't cover the biology/chemistry safety-relevant suite
(WMDP, LAB-Bench, MMLU Pro Biology/Chemistry, BioLP-bench, GPQA Diamond
Biology/Chemistry, plus the GPQA Main and MMLU Biology/Chemistry
sub-splits). The upstream-of-record is:

> Dev et al. (2025), *Toward Comprehensive Benchmarking of the Biological
> Knowledge of Frontier Large Language Models*, RAND Corporation
> techreport **RR-A3797-1** —
> https://www.rand.org/pubs/research_reports/RRA3797-1.html

RAND ships the data as PDF tables inside the report rather than a
machine-readable CSV. The General-Purpose-AI-Policy-Lab "benchmark-
forecasting" repo published a digitized copy. This section pulls that
file (cached into the snapshot folder for provenance) and reshapes it
wide → long.

The CSV's first two rows hold human baselines (per-benchmark expert
score + description) — those are routed to `output/human_baselines.csv`
in section 08 rather than into the model-scores table.


In [5]:
# Original source-of-record: RAND techreport (PDF tables).
# We pull the digitized CSV maintained by the General-Purpose-AI-Policy-Lab.
RAND_URL = (
    "https://raw.githubusercontent.com/General-Purpose-AI-Policy-Lab/"
    "benchmark-forecasting/main/Data/benchmark_data_RAND/benchmark_scores_RAND.csv"
)
RAND_SOURCE_LABEL = "RAND (Dev et al. 2025) — RR-A3797-1"
RAND_SOURCE_LINK  = "https://www.rand.org/pubs/research_reports/RRA3797-1.html"

# Wide-format column → canonical benchmark name
RAND_BENCH_MAP = {
    "Bio: BioLP-bench":             "BioLP-bench",
    "Bio: LAB-Bench Cloning":       "LAB-Bench Cloning",
    "Bio: LAB-Bench LitQA2":        "LAB-Bench LitQA2",
    "Bio: LAB-Bench SeqQA":         "LAB-Bench SeqQA",
    "Bio: LAB-Bench Protocol":      "LAB-Bench Protocol",
    "Bio: GPQA Diamond Biology":    "GPQA Diamond Biology",
    "Bio: GPQA Main Biology":       "GPQA Main Biology",
    "Bio: MMLU Biology":            "MMLU Biology",
    "Bio: WMDP Biology":            "WMDP Biology",
    "Bio: MMLU Pro Biology":        "MMLU Pro Biology",
    "Chem: GPQA Main Chemistry":    "GPQA Main Chemistry",
    "Chem: GPQA Diamond Chemistry": "GPQA Diamond Chemistry",
    "Chem: MMLU Chemistry":         "MMLU Chemistry",
    "Chem: WMDP Chemistry":         "WMDP Chemistry",
    "Chem: MMLU Pro Chemistry":     "MMLU Pro Chemistry",
}

rand_path = LATEST_SNAPSHOT / "rand_benchmark_scores.csv"
if not rand_path.exists():
    print(f"Downloading {RAND_URL}")
    urllib.request.urlretrieve(RAND_URL, rand_path)
    print(f"  -> {rand_path}")
else:
    print(f"RAND CSV already in snapshot — re-using.")

rand_raw = pd.read_csv(rand_path)

# First two rows: human baseline score + description. Save aside.
human_scores = rand_raw.iloc[0]
human_descrs = rand_raw.iloc[1]
rand_models  = rand_raw.iloc[2:].copy()

# Wide → long, restrict to our mapped columns, scale 0-100 -> 0-1
rand_long = rand_models.melt(id_vars=["Model"], var_name="bench_col", value_name="score_raw")
rand_long = rand_long[rand_long["bench_col"].isin(RAND_BENCH_MAP)].copy()
rand_long["benchmark"]     = rand_long["bench_col"].map(RAND_BENCH_MAP)
rand_long["score"]         = pd.to_numeric(rand_long["score_raw"], errors="coerce") / 100.0
rand_long["model_version"] = rand_long["Model"]
rand_long["release_date"]  = pd.NaT
rand_long["organization"]  = ""
rand_long["source"]        = RAND_SOURCE_LABEL
rand_long["source_link"]   = RAND_SOURCE_LINK
rand_long["_source_file"]  = "rand_benchmark_scores.csv"
rand_long["_source_row"]   = rand_long.index
rand_long["_score_col"]    = rand_long["bench_col"]
rand_long = rand_long.dropna(subset=["score"])

keep_cols = ["model_version", "score", "benchmark", "release_date", "organization",
             "source", "source_link", "_source_file", "_source_row", "_score_col"]
rand_long = rand_long[keep_cols]

# Range check
oor = rand_long[(rand_long["score"] < 0) | (rand_long["score"] > 1)]
if len(oor):
    raise ValueError(f"{len(oor)} RAND scores outside [0,1] — check the /100 divisor")

print(f"\nRAND model scores: {len(rand_long):,} rows × {rand_long['benchmark'].nunique()} benchmarks × {rand_long['model_version'].nunique()} models")
print(f"Score range: {rand_long['score'].min():.3f} – {rand_long['score'].max():.3f}")

# Append to scores_long produced in section 03
scores_long = pd.concat([scores_long, rand_long], ignore_index=True, sort=False)
print(f"Total scores_long after RAND merge: {len(scores_long):,} rows × {scores_long['benchmark'].nunique()} benchmarks")

# Re-write the intermediate CSV with the appended RAND rows
scores_long.to_csv(INTERMEDIATE_DIR / "02_scores_long.csv", index=False)

# RAND human baselines — passed to section 08 via the RAND_HUMANS variable
RAND_HUMANS = pd.DataFrame([
    {
        "benchmark": RAND_BENCH_MAP[c],
        "group":     "Domain Expert",
        "score":     pd.to_numeric(human_scores.get(c), errors="coerce") / 100.0,
        "note":      str(human_descrs.get(c, "")),
        "source":    RAND_SOURCE_LINK,
    }
    for c in RAND_BENCH_MAP
])
RAND_HUMANS = RAND_HUMANS.dropna(subset=["score"]).reset_index(drop=True)
print(f"\nRAND human baselines extracted: {len(RAND_HUMANS)} rows (for section 08)")
print(RAND_HUMANS.to_string(index=False))


RAND CSV already in snapshot — re-using.

RAND model scores: 565 rows × 15 benchmarks × 40 models
Score range: 0.010 – 0.922
Total scores_long after RAND merge: 4,054 rows × 60 benchmarks

RAND human baselines extracted: 13 rows (for section 08)
             benchmark         group  score                         note                                                    source
           BioLP-bench Domain Expert  0.384 Bachelor's w/ lab experience https://www.rand.org/pubs/research_reports/RRA3797-1.html
     LAB-Bench Cloning Domain Expert  0.600                In-domain PhD https://www.rand.org/pubs/research_reports/RRA3797-1.html
      LAB-Bench LitQA2 Domain Expert  0.700                In-domain PhD https://www.rand.org/pubs/research_reports/RRA3797-1.html
       LAB-Bench SeqQA Domain Expert  0.780                In-domain PhD https://www.rand.org/pubs/research_reports/RRA3797-1.html
    LAB-Bench Protocol Domain Expert  0.790                In-domain PhD https://www.rand.org/pubs/

## 04 — Canonicalize names

Apply `canonical/benchmark_names.csv` and `canonical/model_aliases.csv`
(both `variant → canonical → note`). If either is missing, it's auto-seeded
with identity rows from what's in the data — edit to rename variants.

Every rename gets recorded in `output/name_changes.csv`. Models that
have no row in `model_aliases.csv` (= new since the last canonicalization
review) get flagged in `output/missing_models.csv`.


In [6]:
NAMES_CSV   = CANONICAL_DIR / "benchmark_names.csv"
ALIASES_CSV = CANONICAL_DIR / "model_aliases.csv"

# Auto-seed on first run from observed variants
if not NAMES_CSV.exists():
    seed = (scores_long[["benchmark"]].drop_duplicates().sort_values("benchmark")
            .rename(columns={"benchmark": "variant"}))
    seed["canonical"] = seed["variant"]
    seed["note"]      = ""
    seed.to_csv(NAMES_CSV, index=False)
    print(f"Seeded {NAMES_CSV.name} with {len(seed)} identity rows.")

if not ALIASES_CSV.exists():
    seed = (scores_long[["model_version"]].drop_duplicates().sort_values("model_version")
            .rename(columns={"model_version": "variant"}))
    seed["canonical"] = seed["variant"]
    seed["note"]      = ""
    seed.to_csv(ALIASES_CSV, index=False)
    print(f"Seeded {ALIASES_CSV.name} with {len(seed)} identity rows.")

bench_df = pd.read_csv(NAMES_CSV)
alias_df = pd.read_csv(ALIASES_CSV)
bench_map = dict(zip(bench_df["variant"], bench_df["canonical"]))
model_map = dict(zip(alias_df["variant"], alias_df["canonical"]))

named = scores_long.copy()
named["benchmark_raw"]     = named["benchmark"]
named["model_version_raw"] = named["model_version"]
named["benchmark"]     = named["benchmark"].map(lambda b: bench_map.get(b, b))
named["model_version"] = named["model_version"].map(lambda m: model_map.get(m, m))

# DROP sentinel: an empty canonical in model_aliases.csv means "remove from dataset".
# Used for uncensored or task-specific fine-tunes whose scores are not informative
# of a unified capability axis (Bio Llama, Hermes 3, Unsafety Llama).
drop_models = set(alias_df.loc[alias_df["canonical"].fillna("") == "", "variant"])
if drop_models:
    drop_mask = named["model_version_raw"].isin(drop_models)
    n_dropped = int(drop_mask.sum())
    n_models  = named.loc[drop_mask, "model_version_raw"].nunique()
    named = named[~drop_mask].reset_index(drop=True)
    print(f"DROP rule: removed {n_dropped} rows from {n_models} models flagged in alias map:")
    for m in sorted(drop_models):
        note = alias_df.loc[alias_df["variant"]==m, "note"].iloc[0]
        print(f"  - {m!r}: {note}")
else:
    print("No DROP-tagged models in alias map.")


name_changes = named[
    (named["benchmark"] != named["benchmark_raw"]) |
    (named["model_version"] != named["model_version_raw"])
][["model_version_raw", "model_version", "benchmark_raw", "benchmark", "_source_file"]].drop_duplicates()
name_changes.to_csv(OUTPUT_DIR / "name_changes.csv", index=False)
print(f"Wrote {len(name_changes)} rename records -> output/name_changes.csv")

# Models that have no entry in the alias map (= new since last review)
alias_variants = set(alias_df["variant"].unique())
new_models = sorted(set(named["model_version_raw"].unique()) - alias_variants)
pd.DataFrame({"model_version": new_models}).to_csv(OUTPUT_DIR / "missing_models.csv", index=False)
print(f"{len(new_models)} models not in alias map -> output/missing_models.csv")

out_path = INTERMEDIATE_DIR / "03_scores_named.csv"
named.to_csv(out_path, index=False)
print(f"Wrote {out_path}")


DROP rule: removed 45 rows from 3 models flagged in alias map:
  - 'Llama 3.1 405b (Bio Llama)': DROP — RAND-specific bio fine-tune; confounds capability with task-specific post-training
  - 'Llama 3.1 405b (Hermes 3)': DROP — NousResearch chat fine-tune; capability axis ill-defined relative to base Instruct
  - 'Unsafety Llama': DROP — uncensored fine-tune; bio/chem scores reflect willingness to answer, not capability
Wrote 515 rename records -> output/name_changes.csv
0 models not in alias map -> output/missing_models.csv
Wrote ~/Desktop/ECI_Bayesian/data/pipeline/intermediate/03_scores_named.csv


## 05 — Dedup

Two passes:

1. **Exact bitwise duplicates** on `(model_version, benchmark, score)` —
   this is the load-bearing case (e.g. EnigmaEval 47×2 from CLAUDE.md).
   Always dropped.
2. **Disagreeing duplicates** on `(model_version, benchmark)` only — these
   shouldn't appear in practice, but if they do, resolve by `DEDUP_POLICY`
   (currently `"max"`). Every conflict goes to `output/duplicates_report.csv`
   with all original scores so the resolution is auditable.


In [7]:
before = len(named)

# Pass 1: exact duplicates
exact_keys = ["model_version", "benchmark", "score"]
n_exact = named.duplicated(subset=exact_keys, keep="first").sum()
deduped = named.drop_duplicates(subset=exact_keys, keep="first").copy()
print(f"Exact duplicates removed: {n_exact}  (before={before}, after={len(deduped)})")

# Pass 2: disagreeing duplicates on (model_version, benchmark)
disagree = (
    deduped.groupby(["model_version", "benchmark"])
    .agg(n=("score", "size"), score_min=("score", "min"), score_max=("score", "max"),
         sources=("_source_file", lambda s: ", ".join(sorted(set(s)))))
    .reset_index()
    .query("n > 1")
    .assign(spread=lambda d: d["score_max"] - d["score_min"])
    .sort_values("spread", ascending=False)
)

print(f"Disagreeing (model, benchmark) groups: {len(disagree)}")
if len(disagree):
    print(disagree.head(20).to_string(index=False))

if DEDUP_POLICY == "max":
    deduped = (deduped.sort_values("score")
                      .drop_duplicates(subset=["model_version", "benchmark"], keep="last"))
else:
    raise ValueError(f"Unknown DEDUP_POLICY: {DEDUP_POLICY}")

disagree.to_csv(OUTPUT_DIR / "duplicates_report.csv", index=False)
print(f"\nWrote output/duplicates_report.csv")
print(f"Final after dedup: {len(deduped):,} rows")

deduped.to_csv(INTERMEDIATE_DIR / "04_scores_deduped.csv", index=False)


Exact duplicates removed: 272  (before=4009, after=3737)
Disagreeing (model, benchmark) groups: 278
                    model_version            benchmark  n  score_min  score_max                    sources  spread
                  Llama-2-70b-hf                 GSM8K  5     0.1330     0.6960         gsm8k_external.csv  0.5630
      gemini-3-deep-think-preview            ARC-AGI-2  2     0.4514     0.8458     arc_agi_2_external.csv  0.3944
         Meta-Llama-3-8B-Instruct           OpenBookQA  2     0.4500     0.8260  open_book_qa_external.csv  0.3760
                 Llama-2-13b-chat                GSM8K  2     0.0270     0.3690         gsm8k_external.csv  0.3420
           gemini-3-flash-preview            ARC-AGI-2  4     0.0125     0.3361     arc_agi_2_external.csv  0.3236
                        PaLM 540B            ARC (AI2)  3     0.5300     0.8520       arc_ai2_external.csv  0.3220
              Baichuan-2-13B-Base                GSM8K  2     0.2210     0.5280         gsm8k_e

## 06 — Attach metadata (category + lower_bound)

Left-join `canonical/benchmark_metadata.csv` on the canonical benchmark
name. Seeded on initial setup from the existing
`data/processed/benchmarks_merged.csv` (56 benchmarks × constant per-benchmark
`category` and `lower_bound`). When new benchmarks appear, the join leaves
those fields NaN and they're listed below — add rows to the metadata file
by hand.

`category` powers `data.py`'s `drop_categories=True` sensitivity run.
`lower_bound` is carried as schema passthrough (the model doesn't read it).


In [8]:
META_CSV = CANONICAL_DIR / "benchmark_metadata.csv"
meta = pd.read_csv(META_CSV)
print(f"Loaded {len(meta)} metadata rows from {META_CSV.name}")

merged = deduped.merge(
    meta[["benchmark", "category", "lower_bound"]],
    on="benchmark", how="left",
)

missing_meta = sorted(merged[merged["category"].isna()]["benchmark"].unique())
if missing_meta:
    print(f"\n{len(missing_meta)} benchmark(s) missing metadata — please add rows to {META_CSV.name}:")
    for b in missing_meta:
        n = (merged["benchmark"] == b).sum()
        print(f"  - {b} ({n} rows)")
else:
    print("All benchmarks have metadata.")


Loaded 75 metadata rows from benchmark_metadata.csv
All benchmarks have metadata.


## 07 — Optional filter

`data.py` already owns date / min-observation / category-drop filters; we
don't pre-filter on those. The one optional filter here is the curated
exclusion list (`data/curated/excluded_benchmarks.txt`), off by default to
match the canonical broad-index config.


In [9]:
EXCL_TXT = CURATED_DIR / "excluded_benchmarks.txt"

filtered = merged.copy()
if APPLY_CURATED_EXCLUSIONS and EXCL_TXT.exists():
    excl = [
        line.strip()
        for line in EXCL_TXT.read_text().splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]
    before = len(filtered)
    filtered = filtered[~filtered["benchmark"].isin(excl)]
    print(f"Applied curated exclusions ({len(excl)} benchmarks): "
          f"before={before}, after={len(filtered)}")
else:
    print(f"APPLY_CURATED_EXCLUSIONS={APPLY_CURATED_EXCLUSIONS} — no filtering applied "
          f"(before={len(filtered)}, after={len(filtered)})")


Applied curated exclusions (8 benchmarks): before=3213, after=2560


## 08 — Humans

Read `data/curated/human_baselines.csv` (the project's authoritative file:
`[benchmark, group, score, note, source]`). Apply the section-04
canonicalization map to human-side benchmark spellings, keep only
benchmarks present in the fitted set, clip into `[ECI_EPS, 1-ECI_EPS]` so
`data.py`'s open-interval filter doesn't drop saturated rows.


In [10]:
HUMANS_IN = CURATED_DIR / "human_baselines.csv"
humans = pd.read_csv(HUMANS_IN)
print(f"Loaded {len(humans)} human-baseline rows from {HUMANS_IN.name}")

humans["benchmark"] = humans["benchmark"].map(lambda b: bench_map.get(b, b))

# Append RAND bio/chem human baselines if section 03b extracted them
if "RAND_HUMANS" in dir():
    humans = pd.concat([humans, RAND_HUMANS], ignore_index=True, sort=False)
    print(f"Appended {len(RAND_HUMANS)} RAND human-baseline rows.")

fitted_benchmarks = set(filtered["benchmark"].unique())
before = len(humans)
humans = humans[humans["benchmark"].isin(fitted_benchmarks)].copy()
print(f"Filter to fitted benchmarks: before={before}, after={len(humans)}")

humans["score"] = humans["score"].clip(lower=ECI_EPS, upper=1 - ECI_EPS)

out_humans = OUTPUT_DIR / "human_baselines.csv"
humans.to_csv(out_humans, index=False)
print(f"Wrote {out_humans}")


Loaded 60 human-baseline rows from human_baselines.csv
Appended 13 RAND human-baseline rows.
Filter to fitted benchmarks: before=73, after=49
Wrote ~/Desktop/ECI_Bayesian/data/pipeline/output/human_baselines.csv


## 09 — Assemble + content diff + report

Final `benchmarks_merged.csv` in the exact schema `data.py` expects, then a
content diff vs the current `data/processed/benchmarks_merged.csv` (small
deltas on overlapping `(model, benchmark)` pairs = good), then a
`pipeline_report.md` summary.


In [11]:
# Match data/processed/benchmarks_merged.csv schema exactly
SCHEMA_COLS = [
    "model_version", "score", "release_date", "organization",
    "benchmark", "stderr", "source", "lower_bound", "category",
]

final = filtered.copy()
for col in SCHEMA_COLS:
    if col not in final.columns:
        if col == "stderr":
            final[col] = np.nan
        elif col == "source":
            final[col] = "EpochAI"
        elif col == "organization":
            final[col] = ""
        elif col == "release_date":
            final[col] = pd.NaT
        else:
            final[col] = np.nan

# Internal Epoch CSVs have no "Source" column; default missing source -> "EpochAI"
final["source"] = final["source"].fillna("EpochAI")
final = final[SCHEMA_COLS].copy()

out_merged = OUTPUT_DIR / "benchmarks_merged.csv"
final.to_csv(out_merged, index=False)
print(f"Wrote {len(final):,} rows -> {out_merged}")

# Content diff vs current file
current_path = PROCESSED_DIR / "benchmarks_merged.csv"
if current_path.exists():
    current = pd.read_csv(current_path)
    join_keys = ["model_version", "benchmark"]
    diff = (
        final[join_keys + ["score"]].rename(columns={"score": "score_new"})
        .merge(current[join_keys + ["score"]].rename(columns={"score": "score_old"}), on=join_keys)
        .assign(delta=lambda d: d["score_new"] - d["score_old"])
    )
    diff_sig = (diff[diff["delta"].abs() > 0.01]
                .assign(abs_delta=lambda d: d["delta"].abs())
                .sort_values("abs_delta", ascending=False)
                .drop(columns=["abs_delta"]))
    diff_sig.to_csv(OUTPUT_DIR / "content_diff_vs_current.csv", index=False)
    n_old = len(current.drop_duplicates(subset=join_keys))
    n_new = len(final.drop_duplicates(subset=join_keys))
    n_overlap = len(diff)
    print(f"\nCoverage vs {current_path.name}:")
    print(f"  current pairs: {n_old:,}")
    print(f"  new pairs:     {n_new:,}")
    print(f"  overlap:       {n_overlap:,}")
    print(f"  significant deltas (|delta|>0.01): {len(diff_sig):,}")
else:
    diff_sig = pd.DataFrame()
    n_old = n_new = n_overlap = 0
    print(f"\nNo current file at {current_path} to diff against.")


Wrote 2,560 rows -> ~/Desktop/ECI_Bayesian/data/pipeline/output/benchmarks_merged.csv

Coverage vs benchmarks_merged.csv:
  current pairs: 2,537
  new pairs:     2,560
  overlap:       1,502
  significant deltas (|delta|>0.01): 156


In [12]:
# Pipeline report
prov = json.loads(prov_path.read_text())
lines = []
lines.append(f"# Pipeline report — {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
lines.append("")
lines.append(f"- **Snapshot:** `{LATEST_SNAPSHOT.name}`")
lines.append(f"- **Epoch ZIP sha256:** `{prov['sha256']}`")
lines.append(f"- **Fetched at:** {prov['fetched_at']}")
lines.append("")
lines.append("## Row counts")
lines.append("")
lines.append(f"- After load:           {len(scores_long):,}")
lines.append(f"- After dedup:          {len(deduped):,}")
lines.append(f"- After metadata join:  {len(merged):,}")
lines.append(f"- After optional filter:{len(filtered):,}")
lines.append(f"- Final:                {len(final):,}")
lines.append("")
lines.append("## Distinct counts")
lines.append("")
lines.append(f"- Benchmarks: {final['benchmark'].nunique()}")
lines.append(f"- Models:     {final['model_version'].nunique()}")
lines.append("")
if n_old:
    lines.append("## Coverage vs current `data/processed/benchmarks_merged.csv`")
    lines.append("")
    lines.append(f"- Overlapping (model, benchmark) pairs: {n_overlap:,}")
    lines.append(f"- Current file pairs: {n_old:,}")
    lines.append(f"- New file pairs:     {n_new:,}")
    lines.append(f"- Significant score deltas (|Δ|>0.01): {len(diff_sig):,}")
    lines.append("")
    if len(diff_sig):
        lines.append("### Top 10 score changes")
        lines.append("")
        for _, r in diff_sig.head(10).iterrows():
            lines.append(f"- `{r['model_version']}` on `{r['benchmark']}`: "
                         f"{r['score_old']:.3f} → {r['score_new']:.3f} (Δ={r['delta']:+.3f})")
        lines.append("")
if missing_meta:
    lines.append("## Benchmarks missing metadata")
    lines.append("")
    lines.append(f"Add rows to `canonical/benchmark_metadata.csv` for:")
    for b in missing_meta:
        lines.append(f"- {b}")
    lines.append("")
if new_models:
    lines.append(f"## New models since last canonicalization ({len(new_models)})")
    lines.append("")
    lines.append("See `output/missing_models.csv`. Add canonical aliases in "
                 "`canonical/model_aliases.csv` and re-run section 04.")
    lines.append("")

report_path = OUTPUT_DIR / "pipeline_report.md"
report_path.write_text("\n".join(lines))
print(f"Wrote {report_path}")


Wrote ~/Desktop/ECI_Bayesian/data/pipeline/output/pipeline_report.md
